In [1]:
import os
import re
import glob
import pandas as pd

# --- CONFIGURAÇÃO ---
DIR_ENTRADA = "datasets entrada"       # Pasta com CSVs originais
DIR_SAIDA = "datasets tratados"        # Pasta saída

# Colunas usadas para identificar duplicatas
# Se a combinação destas colunas se repetir, a linha será removida
COLS_DUPLICATAS = ['DATA', 'CONSUMO_CARGA_ACL']

# Coluna usada para separar os arquivos
COL_AGRUPAMENTO = "SIGLA_PERFIL_AGENTE"
# --------------------

def nome_arquivo_seguro(texto: str) -> str:
    """Normaliza nome para arquivo (remove/transforma caracteres problemáticos)."""
    texto = str(texto).strip().replace(" ", "_")
    return re.sub(r"[^A-Za-z0-9._\-]+", "-", texto)

def processar_dados():
    # 1. PREPARAÇÃO
    os.makedirs(DIR_SAIDA, exist_ok=True)
    arquivos_csv = glob.glob(os.path.join(DIR_ENTRADA, "*.csv"))

    if not arquivos_csv:
        print(f" Nenhum arquivo encontrado em '{DIR_ENTRADA}'")
        return

    print(f"{len(arquivos_csv)} arquivos encontrados.")

    # 2. LEITURA E CONCATENAÇÃO
    try:
        # Lê todos os CSVs e junta em um único DataFrame
        df = pd.concat([pd.read_csv(arq) for arq in arquivos_csv], ignore_index=True)
    except Exception as e:
        print(f"Erro ao ler arquivos: {e}")
        return

    # 3. VERIFICAÇÃO DE COLUNAS
    cols_necessarias = [COL_AGRUPAMENTO, 'DATA'] + COLS_DUPLICATAS
    for col in cols_necessarias:
        if col not in df.columns:
            print(f"Erro: A coluna '{col}' não existe nos dados lidos.")
            return

    # 4. TRATAMENTO DE DATA
    df["DATA"] = pd.to_datetime(df["DATA"], errors="coerce", format="%Y-%m-%d")
    
    # Remove linhas onde a data não pôde ser convertida (opcional, mas recomendado)
    df = df.dropna(subset=["DATA"])

    # 5. PROCESSAMENTO POR SIGLA (AGRUPAMENTO + LIMPEZA)
    
    # Ordena o DF geral para garantir consistência
    df.sort_values(by=[COL_AGRUPAMENTO, "DATA"], kind="stable", inplace=True)

    contagem_arquivos = 0

    for sigla, grupo in df.groupby(COL_AGRUPAMENTO):
        # A. Ordena por data (garante que manteremos o primeiro registro cronológico em caso de duplicata)
        grupo = grupo.sort_values(by="DATA", kind="stable")
        
        qtd_antes = len(grupo)
        
        # B. Remove duplicatas (Lógica do seu segundo script)
        grupo_limpo = grupo.drop_duplicates(subset=COLS_DUPLICATAS, keep='first')
        
        qtd_depois = len(grupo_limpo)
        qtd_removida = qtd_antes - qtd_depois

        # C. Salva o arquivo
        nome_arq = nome_arquivo_seguro(f"{sigla}.csv")
        caminho_final = os.path.join(DIR_SAIDA, nome_arq)
        
        grupo_limpo.to_csv(caminho_final, index=False)

    print(f"\nProcessamento concluído.")

if __name__ == "__main__":
    processar_dados()

2 arquivos encontrados.

Processamento concluído.


In [ ]:
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates 

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping

# ### NOVO: Import para o ARIMA ###
import pmdarima as pm

# ========================
# Configurações principais
# ========================
DATASET_GLOB  = None 
MAX_FILES     = 5
DATASET_PATHS = [
    # "datasets_tratados/ALBRAS.csv",
    # "datasets_tratados/SOUTH32.csv",
    # "datasets_tratados/CSN - UPV.csv",
    # "datasets_tratados/KLABIN PUMA CL.csv",
    # "datasets_tratados/CVRD SE 2.csv",
    "datasets tratados/2RC IND EMBALAGENS.csv",
    "datasets tratados/AACD - IBIRAPUERA.csv",
    "datasets tratados/AAK DO BRASIL CL 514.csv",
]

DATE_COL     = "DATA"
TARGET_COL   = "CONSUMO_CARGA_ACL"
GROUP_COL    = None
GROUP_VALUE  = None

# Opções de preparação
RESAMPLE_RULE = None
AGG_FUNC      = "mean" 

# Hiperparâmetros do modelo
WINDOW     = 60  
EPOCHS     = 100
BATCH_SIZE = 32
TEST_SPLIT = 0.20
SEED       = 42

# Caminho para salvar gráficos e resultados
save_folder_path = './resultados_30D'
os.makedirs(save_folder_path, exist_ok=True)

# Define as sementes (seeds) para reprodutibilidade
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Resolve DATASET_PATHS se None
if DATASET_PATHS is None:
    DATASET_PATHS = sorted(glob(DATASET_GLOB))[:MAX_FILES]
    if not DATASET_PATHS:
        raise FileNotFoundError(f"Nenhum CSV encontrado em: {DATASET_GLOB}")
    if len(DATASET_PATHS) > MAX_FILES:
        DATASET_PATHS = DATASET_PATHS[:MAX_FILES]

print("Arquivos selecionados:")
for p in DATASET_PATHS:
    print(" -", p)

# ================
# Seção de Funções
# ================

def prepare_series(df, date_col, target_col, resample_rule, agg_func):
    """
    Prepara a série temporal: converte datas, remove NAs,
    re-amostra (resample) se necessário, e interpola.
    """
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).sort_values(date_col)

    rule = resample_rule
    if rule is None:
        diffs = df[date_col].diff().dropna().dt.days
        if len(diffs) and diffs.median() <= 1.5: rule = 'D'
        else: rule = 'MS'

    ts = (
        df.set_index(date_col)[target_col]
          .astype(float)
          .resample(rule)
          .agg(agg_func)
          .sort_index()
          .interpolate(method="time")
          .bfill().ffill()
    )
    return ts, rule

def create_windows(data, window_size):
    """
    Cria as janelas deslizantes (X) e os alvos (y) a partir dos dados.
    """
    X, y = [], []
    for i in range(len(data) - window_size - 1):
        X.append(data[i:(i + window_size)])
        y.append(data[i + window_size])
        
    X = np.array(X)
    y = np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))
    return X, y

def create_rnn_model(model_type, input_shape):
    """
    Cria um modelo RNN (LSTM ou GRU) com uma arquitetura padrão.
    """
    model = Sequential()
    if model_type.upper() == 'LSTM':
        model.add(LSTM(50, input_shape=input_shape))
    elif model_type.upper() == 'GRU':
        model.add(GRU(50, input_shape=input_shape))
    else:
        raise ValueError("Tipo de modelo não suportado: 'lstm' ou 'gru'")
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def train_and_evaluate(model_type, X_train, y_train, X_test, y_test_inv, scaler, model_save_path):
    """
    Função completa para criar, treinar e avaliar um modelo RNN.
    (y_test_inv é o conjunto de teste real, já na escala original)
    """
    print("\n" + "-"*30)
    print(f"Treinando {model_type.upper()}...")
    
    input_shape = (X_train.shape[1], X_train.shape[2])
    model = create_rnn_model(model_type, input_shape)
    
    early_stopping = EarlyStopping(
        monitor='val_loss',  
        patience=10,         
        restore_best_weights=True
    )
    
    model.fit(X_train, y_train,
              epochs=EPOCHS, batch_size=BATCH_SIZE,
              verbose=0, # Reduzindo o output
              validation_split=0.1, 
              callbacks=[early_stopping])
    
    model.save(model_save_path)
    print(f"Modelo {model_type.upper()} salvo em: {model_save_path}")

    y_pred_scaled = model.predict(X_test, verbose=0)
    y_pred_inv = scaler.inverse_transform(y_pred_scaled)

    rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
    mae = mean_absolute_error(y_test_inv, y_pred_inv)
    mape = mean_absolute_percentage_error(y_test_inv, y_pred_inv)
    accuracy = (1.0 - mape) * 100
    
    print(f"RMSE {model_type.upper()}: {rmse:.4f}")
    print(f"MAE {model_type.upper()}: {mae:.4f}")
    print(f"Accuracy {model_type.upper()}: {accuracy:.2f}%")

    metrics = {
        f"RMSE_{model_type.upper()}": rmse,
        f"MAE_{model_type.upper()}": mae,
        f"MAPE_{model_type.upper()}": mape,
        f"ACCURACY_{model_type.upper()} (%)": accuracy,
    }
    return y_pred_inv, metrics

# ### Funções de plotagem individual ###

def plot_and_save_single_model(dates, y_true, y_pred, model_name, title_prefix, target_col, save_path):
    """
    Plota e salva o gráfico de resultados para um único modelo (Real vs. Predito).
    """
    plt.figure(figsize=(12, 6))
    
    # Pedido: Real = azul, Predito = laranja, linhas contínuas
    plt.plot(dates, y_true, label='Real')
    plt.plot(dates, y_pred, label= 'Predito')
    
    plt.title(f'{title_prefix} - {model_name}')
    plt.xlabel('Data')
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.gcf().autofmt_xdate()
    plt.ylabel(f'{"Consumo Diário (MWh)"}') 
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_and_save_single_model_zoomed(dates, y_true, y_pred, model_name, title_prefix, target_col, save_path, n_points=30):
    """
    Plota e salva um gráfico focado nos últimos N pontos para um único modelo.
    """
    n_points = min(n_points, len(y_true))
    
    dates_slice = dates[-n_points:]
    y_true_slice = y_true[-n_points:]
    y_pred_slice = y_pred[-n_points:] # Slice dos dados preditos

    plt.figure(figsize=(12, 6))
    
    # Pedido: Real = azul, Predito = laranja, linhas contínuas
    plt.plot(dates_slice, y_true_slice, label=f'Real')
    plt.plot(dates_slice, y_pred_slice, label=f'Predito')
    
    plt.title(f'{title_prefix} - {model_name}')
    plt.xlabel('Data')
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    if n_points <= 45: plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=5))
    else: plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.gcf().autofmt_xdate()
    plt.ylabel(f'{"Consumo Diário (MWh)"}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()


# ==========================
# Loop Principal de Execução
# ==========================
results = [] 

for path in DATASET_PATHS:
    name = os.path.basename(path)
    clean_name = name.replace(".csv", "")
    print("\n" + "="*80)
    print(f"Processando: {name}")

    try:
        # 1) Carregar
        df = pd.read_csv(path)
        
        # 2) Filtro por grupo
        if GROUP_COL and GROUP_VALUE:
            df = df[df[GROUP_COL] == GROUP_VALUE].copy()
            if df.empty:
                print(f"Sem linhas para {GROUP_COL} == {GROUP_VALUE}. Pulando.")
                continue

        # 3) Série preparada
        ts, rule_used = prepare_series(df, DATE_COL, TARGET_COL, RESAMPLE_RULE, AGG_FUNC)
        print(f"Frequência detectada/definida: {rule_used}")
        print(f"Início: {ts.index.min().date()} | Fim: {ts.index.max().date()} | n={len(ts)}")

        # 4) Divisão de Dados
        series_values = ts.values.reshape(-1, 1)
        n = len(series_values)
        
        # ### Verificação de tamanho mínimo ###
        # Precisa de dados suficientes para teste (WINDOW) + treino (pelo menos 1 janela)
        if n < (2 * WINDOW) + 10: 
            print(f"Série muito curta (n={n}) para janela de {WINDOW}. Pulando.")
            continue
            
        n_test = int(np.floor(n * TEST_SPLIT))
        n_train = n - n_test
        
        # Se n_test for muito pequeno, ajusta para garantir que possua o mínimo
        if n_test <= WINDOW + 1:
            n_test = WINDOW + 2 # Mínimo para ter 1 ponto de teste no DL
            n_train = n - n_test
            if n_train < WINDOW + 1: # Checagem final
                 print(f"Não há dados suficientes para treino e teste. Pulando.")
                 continue
        
        # Dados para ARIMA (usando a pd.Series original)
        train_ts = ts.iloc[:n_train]
        test_ts = ts.iloc[n_train:] # O conjunto de teste completo

        # Dados para DL (numpy arrays)
        train_data = series_values[:n_train]
        test_data = series_values[n_train:]
        
        # 5) ### Modelo ARIMA (D=1) ###
        
        # --- Lógica de Sazonalidade ---
        if rule_used == 'D':
            m_period = 7  # 7 dias = Sazonalidade semanal
            seasonal_setting = True
            D_force = 1   # <-- Força a diferenciação sazonal
            print(f"ARIMA: Frequência Diária. Forçando sazonalidade semanal (m=7, D=1).")
        elif rule_used == 'MS':
            m_period = 12 # 12 meses = Sazonalidade anual
            seasonal_setting = True
            D_force = 1   # <-- Força a diferenciação sazonal
            print(f"ARIMA: Frequência Mensal. Forçando sazonalidade anual (m=12, D=1).")
        else:
            m_period = 1
            seasonal_setting = False
            D_force = 0   # <-- Sem D se não houver sazonalidade
            print(f"ARIMA: Frequência não padrão. Desligando sazonalidade (m=1, D=0).")
        # --------------------------------

        print("\n" + "-"*30)
        print("Treinando AUTO-ARIMA")
        
        # Treina o auto_arima (D=1)
        arima_model = pm.auto_arima(
            train_ts,
            start_p=1, start_q=1,
            test='adf',           
            max_p=5, max_q=5,
            m=m_period,               # Define o período sazonal
            seasonal=seasonal_setting, # Habilita a sazonalidade
            
            # --- ATUALIZAÇÃO PRINCIPAL ---
            D=D_force,                # <-- D=1 (em vez de D=None)
            # ---------------------------
            
            start_P=1, max_P=2, 
            start_Q=1, max_Q=2,
            
            trace=False,          
            error_action='ignore',
            suppress_warnings=True,
            stepwise=True         # Mantemos o stepwise por enquanto (para velocidade)
        )
        print(f"ARIMA(p,d,q)x(P,D,Q)m selecionado: {arima_model.order}x{arima_model.seasonal_order}")

        # Faz a predição para o período de teste COMPLETO
        y_pred_arima_full = arima_model.predict(n_periods=len(test_ts))
        
        # 6) Preparação de Dados para DL (LSTM/GRU)
        scaler = MinMaxScaler(feature_range=(0, 1))
        train_scaled = scaler.fit_transform(train_data)
        test_scaled = scaler.transform(test_data)
        
        X_train, y_train = create_windows(train_scaled, WINDOW)
        X_test, y_test_scaled = create_windows(test_scaled, WINDOW)

        # 7) ### Definição do Conjunto de Teste comum ###
        # Para uma comparação justa, todos os modelos são avaliados
        # APENAS nos pontos que o LSTM/GRU conseguem predizer.
        
        # O 'y_test' do DL (na escala original)
        y_test_common = scaler.inverse_transform(y_test_scaled.reshape(-1, 1))
        
        # As datas do 'y_test' do DL
        test_start_index = n_train + WINDOW
        test_end_index = test_start_index + len(y_test_scaled)
        y_test_dates_common = ts.index[test_start_index : test_end_index]
        
        if len(y_test_common) == 0:
            print(f"Não há dados de teste suficientes após janelamento DL. Pulando.")
            continue
            
        # 8) ### Alinhar Predições ARIMA ###
        # Seleciona apenas as predições do ARIMA que correspondem as datas do teste comum
        y_pred_arima_common = y_pred_arima_full.loc[y_test_dates_common].values.reshape(-1, 1)

        # 9) Calcular Métricas ARIMA no conjunto comum
        rmse_arima = np.sqrt(mean_squared_error(y_test_common, y_pred_arima_common))
        mae_arima = mean_absolute_error(y_test_common, y_pred_arima_common)
        mape_arima = mean_absolute_percentage_error(y_test_common, y_pred_arima_common)
        accuracy_arima = (1.0 - mape_arima) * 100
        
        arima_metrics = {
            "RMSE_ARIMA": rmse_arima,
            "MAE_ARIMA": mae_arima,
            "MAPE_ARIMA": mape_arima,
            "ACCURACY_ARIMA (%)": accuracy_arima,
        }
        print(f"RMSE ARIMA: {rmse_arima:.4f}")
        print(f"MAE ARIMA: {mae_arima:.4f}")
        print(f"Accuracy ARIMA: {accuracy_arima:.2f}%")

        # 10) Treino e Avaliação DL (LSTM & GRU)
        # Passa o 'y_test_common' para a função de avaliação
        
        lstm_save_path = os.path.join(save_folder_path, f"{clean_name}_lstm.h5")
        y_pred_lstm_inv, lstm_metrics = train_and_evaluate(
            'lstm', X_train, y_train, X_test, y_test_common, scaler, lstm_save_path
        )
        
        gru_save_path = os.path.join(save_folder_path, f"{clean_name}_gru.h5")
        y_pred_gru_inv, gru_metrics = train_and_evaluate(
            'gru', X_train, y_train, X_test, y_test_common, scaler, gru_save_path
        )

        # 11) Armazenar Resultados
        result_data = {
            "csv": name,
            "freq": rule_used,
            "train_samples": len(X_train),
            "test_samples_common": len(y_test_common), # Tamanho do teste comum
        }
        result_data.update(arima_metrics) # Adiciona métricas do ARIMA
        result_data.update(lstm_metrics)  # Adiciona métricas do LSTM
        result_data.update(gru_metrics)   # Adiciona métricas do GRU
        results.append(result_data)

        # 12) ### Plotar gráficos individuais ###
        plot_title_prefix = f'{clean_name}'

        # --- Plot LSTM ---
        plot_path_lstm = os.path.join(save_folder_path, f"{clean_name}_LSTM.png")
        plot_and_save_single_model(
            y_test_dates_common, y_test_common.flatten(), 
            y_pred_lstm_inv.flatten(), 
            "LSTM", plot_title_prefix, TARGET_COL, plot_path_lstm
        )
        plot_path_lstm_zoom = os.path.join(save_folder_path, f"{clean_name}_LSTM_30.png")
        plot_and_save_single_model_zoomed(
            y_test_dates_common, y_test_common.flatten(), 
            y_pred_lstm_inv.flatten(), 
            "LSTM", plot_title_prefix, TARGET_COL, plot_path_lstm_zoom, n_points=30
        )

        # --- Plot GRU ---
        plot_path_gru = os.path.join(save_folder_path, f"{clean_name}_GRU.png")
        plot_and_save_single_model(
            y_test_dates_common, y_test_common.flatten(), 
            y_pred_gru_inv.flatten(), 
            "GRU", plot_title_prefix, TARGET_COL, plot_path_gru
        )
        plot_path_gru_zoom = os.path.join(save_folder_path, f"{clean_name}_GRU_30.png")
        plot_and_save_single_model_zoomed(
            y_test_dates_common, y_test_common.flatten(), 
            y_pred_gru_inv.flatten(), 
            "GRU", plot_title_prefix, TARGET_COL, plot_path_gru_zoom, n_points=30
        )

        # --- Plot ARIMA ---
        plot_path_arima = os.path.join(save_folder_path, f"{clean_name}_ARIMA.png")
        plot_and_save_single_model(
            y_test_dates_common, y_test_common.flatten(), 
            y_pred_arima_common.flatten(), 
            "ARIMA", plot_title_prefix, TARGET_COL, plot_path_arima
        )
        plot_path_arima_zoom = os.path.join(save_folder_path, f"{clean_name}_ARIMA_30.png")
        plot_and_save_single_model_zoomed(
            y_test_dates_common, y_test_common.flatten(), 
            y_pred_arima_common.flatten(), 
            "ARIMA", plot_title_prefix, TARGET_COL, plot_path_arima_zoom, n_points=30
        )
        
    except Exception as e:
        print(f"Falha ao processar {name}: {e}")
        import traceback
        traceback.print_exc() # Imprime o stack trace completo
        continue # Pula para o próximo arquivo

# =================
# Resultados Finais
# =================
if results:
    metrics_df = pd.DataFrame(results)
    
    # ### Colunas atualizadas ###
    out_cols_full = [
        "csv", "freq", 
        "RMSE_ARIMA", "MAE_ARIMA", "MAPE_ARIMA", "ACCURACY_ARIMA (%)",
        "RMSE_LSTM", "MAE_LSTM", "MAPE_LSTM", "ACCURACY_LSTM (%)",
        "RMSE_GRU", "MAE_GRU", "MAPE_GRU", "ACCURACY_GRU (%)",
        "test_samples_common"
    ]
    out_cols_full = [col for col in out_cols_full if col in metrics_df.columns]
    
    print("\n" + "="*80)
    print("=== Tabela de Métricas ===")
    
    metrics_df_display = metrics_df.copy()
    for model_name in ["ARIMA", "LSTM", "GRU"]:
        if f"MAPE_{model_name}" in metrics_df_display:
            metrics_df_display[f"MAPE_{model_name}"] = metrics_df_display[f"MAPE_{model_name}"].map('{:.4f}'.format)
            metrics_df_display[f"ACCURACY_{model_name} (%)"] = metrics_df_display[f"ACCURACY_{model_name} (%)"].map('{:.2f}%'.format)
    
    print(metrics_df_display[out_cols_full].to_string(index=False))
    
    metrics_csv_path = os.path.join(save_folder_path, "metricas_combinadas_arima_lstm_gru.csv")
    metrics_df[out_cols_full].to_csv(metrics_csv_path, index=False)
    print(f"\nArquivo de métricas completo salvo em: {metrics_csv_path}")

    # Tabela de Acurácia
    print("\n" + "="*80)
    print("=== Tabela de Acurácia ===")
    
    out_cols_accuracy = ["csv", "ACCURACY_ARIMA (%)", "ACCURACY_LSTM (%)", "ACCURACY_GRU (%)"]
    out_cols_accuracy = [col for col in out_cols_accuracy if col in metrics_df_display.columns]

    print(metrics_df_display[out_cols_accuracy].to_string(index=False))
    print("="*80)

    # ### Gráfico de barras de RMSE ###
    plt.figure(figsize=(15, 8))
    index = np.arange(len(metrics_df))
    bar_width = 0.25 # Largura da barra
    
    plt.bar(index - bar_width, metrics_df['RMSE_ARIMA'], bar_width, label='ARIMA', color='green')
    plt.bar(index, metrics_df['RMSE_LSTM'], bar_width, label='LSTM', color='blue')
    plt.bar(index + bar_width, metrics_df['RMSE_GRU'], bar_width, label='GRU', color='orange')
    
    plt.xlabel('Dataset', fontsize=16)
    plt.ylabel('RMSE (no conjunto de teste comum)', fontsize=16)
    plt.title('Comparação de RMSE entre os modelos (ARIMA, LSTM, GRU)', fontsize=16)
    plt.xticks(index, metrics_df['csv'], rotation=45, ha='right', fontsize=12)
    plt.legend(prop={'size': 13})
    plt.tight_layout()
    
    rmse_plot_path = os.path.join(save_folder_path, 'RMSE_Comparison_Bar_Chart.png')
    plt.savefig(rmse_plot_path)
    plt.close()
    print(f"Gráfico de barras de RMSE salvo em: {rmse_plot_path}")

else:
    print("\nNenhum resultado para mostrar (todas séries muito curtas ou sem dados).")

Arquivos selecionados:
 - datasets tratados/2RC IND EMBALAGENS.csv
 - datasets tratados/AACD - IBIRAPUERA.csv
 - datasets tratados/AAK DO BRASIL CL 514.csv

Processando: 2RC IND EMBALAGENS.csv
Frequência detectada/definida: D
Início: 2024-03-01 | Fim: 2025-08-31 | n=549
ARIMA: Frequência Diária. Forçando sazonalidade semanal (m=7, D=1).

------------------------------
Treinando AUTO-ARIMA (forçando D=1)...


/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: F

ARIMA(p,d,q)x(P,D,Q)m selecionado: (1, 0, 2)x(0, 1, 1, 7)
RMSE ARIMA (comum): 1.4531
MAE ARIMA (comum): 1.0807
Accuracy ARIMA (comum): -180.84%

------------------------------
Treinando LSTM...


/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Modelo LSTM salvo em: ./graficos_separados/2RC IND EMBALAGENS_lstm.h5
RMSE LSTM: 1.1694
MAE LSTM: 0.8888
Accuracy LSTM: -711.53%

------------------------------
Treinando GRU...


...

/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: F

ARIMA(p,d,q)x(P,D,Q)m selecionado: (2, 0, 1)x(0, 1, 1, 7)
RMSE ARIMA (comum): 5.0848
MAE ARIMA (comum): 3.5905
Accuracy ARIMA (comum): 84.48%

------------------------------
Treinando LSTM...


/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Modelo LSTM salvo em: ./graficos_separados/AAK DO BRASIL CL 514_lstm.h5
RMSE LSTM: 5.6898
MAE LSTM: 3.3826
Accuracy LSTM: 85.10%

------------------------------
Treinando GRU...


Modelo GRU salvo em: ./graficos_separados/AAK DO BRASIL CL 514_gru.h5
RMSE GRU: 5.5998
MAE GRU: 3.4833
Accuracy GRU: 85.69%

=== Tabela de Métricas (Completa) ===
                     csv freq  RMSE_ARIMA  MAE_ARIMA MAPE_ARIMA ACCURACY_ARIMA (%)  RMSE_LSTM  MAE_LSTM MAPE_LSTM ACCURACY_LSTM (%)  RMSE_GRU  MAE_GRU MAPE_GRU ACCURACY_GRU (%)  test_samples_common
  2RC IND EMBALAGENS.csv    D    1.453092   1.080708     2.8084           -180.84%   1.169397  0.888759    8.1153          -711.53%  1.647517 1.273074  10.7828         -978.28%                   48
   AACD - IBIRAPUERA.csv    D    2.596874   2.442360     0.1813             81.87%   1.743156  1.260072    0.1016            89.84%  0.948179 0.731161   0.0557           94.43%                  103
AAK DO BRASIL CL 514.csv    D    5.084823   3.590490     0.1552             84.48%   5.689795  3.382552    0.1490            85.10%  5.599845 3.483315   0.1431           85.69%                  206

Arquivo de métricas completo salvo em: ./gra

In [ ]:
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates 

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping

# ### Import para o ARIMA ###
import pmdarima as pm

# ========================
# Configurações principais
# ========================
DATASET_GLOB  = None 
MAX_FILES     = 5
DATASET_PATHS = [
    # "datasets_tratados/ALBRAS.csv",
    # "datasets_tratados/SOUTH32.csv",
    # "datasets_tratados/CSN - UPV.csv",
    # "datasets_tratados/KLABIN PUMA CL.csv",
    # "datasets_tratados/CVRD SE 2.csv",
    "datasets tratados/2RC IND EMBALAGENS.csv",
    "datasets tratados/AACD - IBIRAPUERA.csv",
    "datasets tratados/AAK DO BRASIL CL 514.csv",
]

DATE_COL     = "DATA"
TARGET_COL   = "CONSUMO_CARGA_ACL"
GROUP_COL    = None
GROUP_VALUE  = None

# Opções de preparação
RESAMPLE_RULE = None
AGG_FUNC      = "mean" 

# Hiperparâmetros do modelo
WINDOW     = 60  
EPOCHS     = 100
BATCH_SIZE = 32
TEST_SPLIT = 0.20
SEED       = 42

# Caminho para salvar gráficos e resultados
save_folder_path = './resultados_90D'
os.makedirs(save_folder_path, exist_ok=True)

# Define as sementes (seeds) para reprodutibilidade
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Resolve DATASET_PATHS se None
if DATASET_PATHS is None:
    DATASET_PATHS = sorted(glob(DATASET_GLOB))[:MAX_FILES]
    if not DATASET_PATHS:
        raise FileNotFoundError(f"Nenhum CSV encontrado em: {DATASET_GLOB}")
    if len(DATASET_PATHS) > MAX_FILES:
        DATASET_PATHS = DATASET_PATHS[:MAX_FILES]

print("Arquivos selecionados:")
for p in DATASET_PATHS:
    print(" -", p)

# ================
# Seção de Funções
# ================

def prepare_series(df, date_col, target_col, resample_rule, agg_func):
    """
    Prepara a série temporal: converte datas, remove NAs,
    re-amostra (resample) se necessário, e interpola.
    """
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).sort_values(date_col)

    rule = resample_rule
    if rule is None:
        diffs = df[date_col].diff().dropna().dt.days
        if len(diffs) and diffs.median() <= 1.5: rule = 'D'
        else: rule = 'MS'

    ts = (
        df.set_index(date_col)[target_col]
          .astype(float)
          .resample(rule)
          .agg(agg_func)
          .sort_index()
          .interpolate(method="time")
          .bfill().ffill()
    )
    return ts, rule

def create_windows(data, window_size):
    """
    Cria as janelas deslizantes (X) e os alvos (y) a partir dos dados.
    """
    X, y = [], []
    for i in range(len(data) - window_size - 1):
        X.append(data[i:(i + window_size)])
        y.append(data[i + window_size])
        
    X = np.array(X)
    y = np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))
    return X, y

def create_rnn_model(model_type, input_shape):
    """
    Cria um modelo RNN (LSTM ou GRU) com uma arquitetura padrão.
    """
    model = Sequential()
    if model_type.upper() == 'LSTM':
        model.add(LSTM(50, input_shape=input_shape))
    elif model_type.upper() == 'GRU':
        model.add(GRU(50, input_shape=input_shape))
    else:
        raise ValueError("Tipo de modelo não suportado: 'lstm' ou 'gru'")
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def train_and_evaluate(model_type, X_train, y_train, X_test, y_test_inv, scaler, model_save_path):
    """
    Função completa para criar, treinar e avaliar um modelo RNN.
    (y_test_inv é o conjunto de teste real, já na escala original)
    """
    print("\n" + "-"*30)
    print(f"Treinando {model_type.upper()}...")
    
    input_shape = (X_train.shape[1], X_train.shape[2])
    model = create_rnn_model(model_type, input_shape)
    
    early_stopping = EarlyStopping(
        monitor='val_loss',  
        patience=10,         
        restore_best_weights=True
    )
    
    model.fit(X_train, y_train,
              epochs=EPOCHS, batch_size=BATCH_SIZE,
              verbose=0, # Reduzindo o output
              validation_split=0.1, 
              callbacks=[early_stopping])
    
    model.save(model_save_path)
    print(f"Modelo {model_type.upper()} salvo em: {model_save_path}")

    y_pred_scaled = model.predict(X_test, verbose=0)
    y_pred_inv = scaler.inverse_transform(y_pred_scaled)

    rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
    mae = mean_absolute_error(y_test_inv, y_pred_inv)
    mape = mean_absolute_percentage_error(y_test_inv, y_pred_inv)
    accuracy = (1.0 - mape) * 100
    
    print(f"RMSE {model_type.upper()}: {rmse:.4f}")
    print(f"MAE {model_type.upper()}: {mae:.4f}")
    print(f"Accuracy {model_type.upper()}: {accuracy:.2f}%")

    metrics = {
        f"RMSE_{model_type.upper()}": rmse,
        f"MAE_{model_type.upper()}": mae,
        f"MAPE_{model_type.upper()}": mape,
        f"ACCURACY_{model_type.upper()} (%)": accuracy,
    }
    return y_pred_inv, metrics

# ### Funções de plotagem individual ###

def plot_and_save_single_model(dates, y_true, y_pred, model_name, title_prefix, target_col, save_path):
    """
    Plota e salva o gráfico de resultados para um único modelo (Real vs. Predito).
    """
    plt.figure(figsize=(12, 6), edgecolor='black', linewidth=1)
    # plt.figure(figsize=(12, 6))
    
    # Real = azul, Predito = laranja, linhas contínuas
    plt.plot(dates, y_true, label='Real')
    plt.plot(dates, y_pred, label= 'Predito')
    
    plt.title(f'{title_prefix} - {model_name}')
    plt.xlabel('Data')
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.gcf().autofmt_xdate()
    plt.ylabel(f'{"Consumo Diário (MWh)"}') 
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_and_save_single_model_zoomed(dates, y_true, y_pred, model_name, title_prefix, target_col, save_path, n_points=30):
    """
    Plota e salva um gráfico focado nos últimos N pontos para um único modelo.
    """
    n_points = min(n_points, len(y_true))
    
    dates_slice = dates[-n_points:]
    y_true_slice = y_true[-n_points:]
    y_pred_slice = y_pred[-n_points:] # Slice dos dados preditos

    plt.figure(figsize=(12, 6), edgecolor='black', linewidth=1)
    
    # Real = azul, Predito = laranja, linhas contínuas
    plt.plot(dates_slice, y_true_slice, label=f'Real')
    plt.plot(dates_slice, y_pred_slice, label=f'Predito')
    
    plt.title(f'{title_prefix} - {model_name}')
    plt.xlabel('Data')
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    if n_points <= 45: plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=5))
    else: plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.gcf().autofmt_xdate()
    plt.ylabel(f'{"Consumo Diário (MWh)"}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()


# ==========================
# Loop Principal de Execução
# ==========================
results = [] 

for path in DATASET_PATHS:
    name = os.path.basename(path)
    clean_name = name.replace(".csv", "")
    print("\n" + "="*80)
    print(f"Processando: {name}")

    try:
        # 1) Carregar
        df = pd.read_csv(path)
        
        # 2) Filtro por grupo
        if GROUP_COL and GROUP_VALUE:
            df = df[df[GROUP_COL] == GROUP_VALUE].copy()
            if df.empty:
                print(f"Sem linhas para {GROUP_COL} == {GROUP_VALUE}. Pulando.")
                continue

        # 3) Série preparada
        ts, rule_used = prepare_series(df, DATE_COL, TARGET_COL, RESAMPLE_RULE, AGG_FUNC)
        print(f"Frequência detectada/definida: {rule_used}")
        print(f"Início: {ts.index.min().date()} | Fim: {ts.index.max().date()} | n={len(ts)}")

        # 4) Divisão de Dados (Unificada)
        series_values = ts.values.reshape(-1, 1)
        n = len(series_values)
        
        # ### Verificação de tamanho mínimo ###
        # Precisa de dados suficientes para teste (WINDOW) + treino (pelo menos 1 janela)
        if n < (2 * WINDOW) + 10: 
            print(f"Série muito curta (n={n}) para janela de {WINDOW}. Pulando.")
            continue
            
        n_test = int(np.floor(n * TEST_SPLIT))
        n_train = n - n_test
        
        # Se n_test for muito pequeno, ajusta para garantir que possua o mínimo
        if n_test <= WINDOW + 1:
            n_test = WINDOW + 2 # Mínimo para ter 1 ponto de teste no DL
            n_train = n - n_test
            if n_train < WINDOW + 1: # Checagem final
                 print(f"Não há dados suficientes para treino e teste. Pulando.")
                 continue
        
        # Dados para ARIMA (usando a pd.Series original)
        train_ts = ts.iloc[:n_train]
        test_ts = ts.iloc[n_train:] # O conjunto de teste completo

        # Dados para DL (numpy arrays)
        train_data = series_values[:n_train]
        test_data = series_values[n_train:]
        
        # 5) ### Modelo ARIMA (D=1) ###
        
        # --- Lógica de Sazonalidade ---
        if rule_used == 'D':
            m_period = 7  # 7 dias = Sazonalidade semanal
            seasonal_setting = True
            D_force = 1   # <-- Força a diferenciação sazonal
            print(f"ARIMA: Frequência Diária. Forçando sazonalidade semanal (m=7, D=1).")
        elif rule_used == 'MS':
            m_period = 12 # 12 meses = Sazonalidade anual
            seasonal_setting = True
            D_force = 1   # <-- Força a diferenciação sazonal
            print(f"ARIMA: Frequência Mensal. Forçando sazonalidade anual (m=12, D=1).")
        else:
            m_period = 1
            seasonal_setting = False
            D_force = 0   # <-- Sem D se não houver sazonalidade
            print(f"ARIMA: Frequência não padrão. Desligando sazonalidade (m=1, D=0).")
        # --------------------------------

        print("\n" + "-"*30)
        print("Treinando AUTO-ARIMA.")
        
        # Treina o auto_arima
        arima_model = pm.auto_arima(
            train_ts,
            start_p=1, start_q=1,
            test='adf',           
            max_p=5, max_q=5,
            m=m_period,               # Define o período sazonal
            seasonal=seasonal_setting, # Habilita a sazonalidade
            
            # --- ATUALIZAÇÃO PRINCIPAL ---
            D=D_force,                # <-- D=1 (em vez de D=None)
            # ---------------------------
            
            start_P=1, max_P=2, 
            start_Q=1, max_Q=2,
            
            trace=False,          
            error_action='ignore',
            suppress_warnings=True,
            stepwise=True         # Mantemos o stepwise por enquanto (para velocidade)
        )
        print(f"ARIMA(p,d,q)x(P,D,Q)m selecionado: {arima_model.order}x{arima_model.seasonal_order}")

        # Faz a prediçãi para o período de teste COMPLETO
        y_pred_arima_full = arima_model.predict(n_periods=len(test_ts))
        
        # 6) Preparação de Dados para DL (LSTM/GRU)
        scaler = MinMaxScaler(feature_range=(0, 1))
        train_scaled = scaler.fit_transform(train_data)
        test_scaled = scaler.transform(test_data)
        
        X_train, y_train = create_windows(train_scaled, WINDOW)
        X_test, y_test_scaled = create_windows(test_scaled, WINDOW)

        # 7) ### Definição do Conjunto de Teste COMUM ###
        # Para uma comparação justa, todos os modelos são avaliados
        # APENAS nos pontos que o LSTM/GRU conseguem prever.
        
        # O 'y_test' do DL (na escala original)
        y_test_common = scaler.inverse_transform(y_test_scaled.reshape(-1, 1))
        
        # As datas do 'y_test' do DL
        test_start_index = n_train + WINDOW
        test_end_index = test_start_index + len(y_test_scaled)
        y_test_dates_common = ts.index[test_start_index : test_end_index]
        
        if len(y_test_common) == 0:
            print(f"Não há dados de teste suficientes após janelamento DL. Pulando.")
            continue
            
        # 8) ### Alinhar Previsões ARIMA ###
        # Seleciona apenas as predições do ARIMA que correspondem as datas do teste comum
        y_pred_arima_common = y_pred_arima_full.loc[y_test_dates_common].values.reshape(-1, 1)

        # 9) Calcular Métricas ARIMA no conjunto comum
        rmse_arima = np.sqrt(mean_squared_error(y_test_common, y_pred_arima_common))
        mae_arima = mean_absolute_error(y_test_common, y_pred_arima_common)
        mape_arima = mean_absolute_percentage_error(y_test_common, y_pred_arima_common)
        accuracy_arima = (1.0 - mape_arima) * 100
        
        arima_metrics = {
            "RMSE_ARIMA": rmse_arima,
            "MAE_ARIMA": mae_arima,
            "MAPE_ARIMA": mape_arima,
            "ACCURACY_ARIMA (%)": accuracy_arima,
        }
        print(f"RMSE ARIMA: {rmse_arima:.4f}")
        print(f"MAE ARIMA: {mae_arima:.4f}")
        print(f"Accuracy ARIMA: {accuracy_arima:.2f}%")

        # 10) Treino e Avaliação DL (LSTM & GRU)
        # Passa o 'y_test_common' para a função de avaliação
        
        lstm_save_path = os.path.join(save_folder_path, f"{clean_name}_lstm.h5")
        y_pred_lstm_inv, lstm_metrics = train_and_evaluate(
            'lstm', X_train, y_train, X_test, y_test_common, scaler, lstm_save_path
        )
        
        gru_save_path = os.path.join(save_folder_path, f"{clean_name}_gru.h5")
        y_pred_gru_inv, gru_metrics = train_and_evaluate(
            'gru', X_train, y_train, X_test, y_test_common, scaler, gru_save_path
        )

        # 11) Armazenar Resultados
        result_data = {
            "csv": name,
            "freq": rule_used,
            "train_samples": len(X_train),
            "test_samples_common": len(y_test_common), # Tamanho do teste comum
        }
        result_data.update(arima_metrics) # Adiciona métricas do ARIMA
        result_data.update(lstm_metrics)  # Adiciona métricas do LSTM
        result_data.update(gru_metrics)   # Adiciona métricas do GRU
        results.append(result_data)

        # 12) ### Plotar gráficos individuais ###
        plot_title_prefix = f'{clean_name}'

        # --- Plot LSTM ---
        plot_path_lstm = os.path.join(save_folder_path, f"{clean_name}_LSTM.png")
        plot_and_save_single_model(
            y_test_dates_common, y_test_common.flatten(), 
            y_pred_lstm_inv.flatten(), 
            "LSTM", plot_title_prefix, TARGET_COL, plot_path_lstm
        )
        plot_path_lstm_zoom = os.path.join(save_folder_path, f"{clean_name}_LSTM_90.png")
        plot_and_save_single_model_zoomed(
            y_test_dates_common, y_test_common.flatten(), 
            y_pred_lstm_inv.flatten(), 
            "LSTM", plot_title_prefix, TARGET_COL, plot_path_lstm_zoom, n_points=90
        )

        # --- Plot GRU ---
        plot_path_gru = os.path.join(save_folder_path, f"{clean_name}_GRU.png")
        plot_and_save_single_model(
            y_test_dates_common, y_test_common.flatten(), 
            y_pred_gru_inv.flatten(), 
            "GRU", plot_title_prefix, TARGET_COL, plot_path_gru
        )
        plot_path_gru_zoom = os.path.join(save_folder_path, f"{clean_name}_GRU_90.png")
        plot_and_save_single_model_zoomed(
            y_test_dates_common, y_test_common.flatten(), 
            y_pred_gru_inv.flatten(), 
            "GRU", plot_title_prefix, TARGET_COL, plot_path_gru_zoom, n_points=90
        )

        # --- Plot ARIMA ---
        plot_path_arima = os.path.join(save_folder_path, f"{clean_name}_ARIMA.png")
        plot_and_save_single_model(
            y_test_dates_common, y_test_common.flatten(), 
            y_pred_arima_common.flatten(), 
            "ARIMA", plot_title_prefix, TARGET_COL, plot_path_arima
        )
        plot_path_arima_zoom = os.path.join(save_folder_path, f"{clean_name}_ARIMA_90.png")
        plot_and_save_single_model_zoomed(
            y_test_dates_common, y_test_common.flatten(), 
            y_pred_arima_common.flatten(), 
            "ARIMA", plot_title_prefix, TARGET_COL, plot_path_arima_zoom, n_points=90
        )
        
    except Exception as e:
        print(f"Falha ao processar {name}: {e}")
        import traceback
        traceback.print_exc() # Imprime o stack trace completo
        continue # Pula para o próximo arquivo

# =================
# Resultados Finais
# =================
if results:
    metrics_df = pd.DataFrame(results)
    
    # ### Colunas atualizadas ###
    out_cols_full = [
        "csv", "freq", 
        "RMSE_ARIMA", "MAE_ARIMA", "MAPE_ARIMA", "ACCURACY_ARIMA (%)",
        "RMSE_LSTM", "MAE_LSTM", "MAPE_LSTM", "ACCURACY_LSTM (%)",
        "RMSE_GRU", "MAE_GRU", "MAPE_GRU", "ACCURACY_GRU (%)",
        "test_samples_common"
    ]
    out_cols_full = [col for col in out_cols_full if col in metrics_df.columns]
    
    print("\n" + "="*80)
    print("=== Tabela de Métricas ===")
    
    metrics_df_display = metrics_df.copy()
    for model_name in ["ARIMA", "LSTM", "GRU"]:
        if f"MAPE_{model_name}" in metrics_df_display:
            metrics_df_display[f"MAPE_{model_name}"] = metrics_df_display[f"MAPE_{model_name}"].map('{:.4f}'.format)
            metrics_df_display[f"ACCURACY_{model_name} (%)"] = metrics_df_display[f"ACCURACY_{model_name} (%)"].map('{:.2f}%'.format)
    
    print(metrics_df_display[out_cols_full].to_string(index=False))
    
    metrics_csv_path = os.path.join(save_folder_path, "metricas_combinadas_arima_lstm_gru.csv")
    metrics_df[out_cols_full].to_csv(metrics_csv_path, index=False)
    print(f"\nArquivo de métricas completo salvo em: {metrics_csv_path}")

    # Tabela de Acurácia simplificada
    print("\n" + "="*80)
    print("=== Tabela de Acurácia ===")
    
    out_cols_accuracy = ["csv", "ACCURACY_ARIMA (%)", "ACCURACY_LSTM (%)", "ACCURACY_GRU (%)"]
    out_cols_accuracy = [col for col in out_cols_accuracy if col in metrics_df_display.columns]

    print(metrics_df_display[out_cols_accuracy].to_string(index=False))
    print("="*80)

    # ### Gráfico de barras de RMSE ###
    plt.figure(figsize=(15, 8))
    index = np.arange(len(metrics_df))
    bar_width = 0.25 # Largura barra
    
    plt.bar(index - bar_width, metrics_df['RMSE_ARIMA'], bar_width, label='ARIMA', color='green')
    plt.bar(index, metrics_df['RMSE_LSTM'], bar_width, label='LSTM', color='blue')
    plt.bar(index + bar_width, metrics_df['RMSE_GRU'], bar_width, label='GRU', color='orange')
    
    plt.xlabel('Dataset', fontsize=16)
    plt.ylabel('RMSE (no conjunto de teste comum)', fontsize=16)
    plt.title('Comparação de RMSE entre os modelos (ARIMA, LSTM, GRU)', fontsize=16)
    plt.xticks(index, metrics_df['csv'], rotation=45, ha='right', fontsize=12)
    plt.legend(prop={'size': 13})
    plt.tight_layout()
    
    rmse_plot_path = os.path.join(save_folder_path, 'RMSE_Comparison_Bar_Chart.png')
    plt.savefig(rmse_plot_path)
    plt.close()
    print(f"Gráfico de barras de RMSE salvo em: {rmse_plot_path}")

else:
    print("\nNenhum resultado para mostrar (todas séries muito curtas ou sem dados).")

Arquivos selecionados:
 - datasets tratados/2RC IND EMBALAGENS.csv
 - datasets tratados/AACD - IBIRAPUERA.csv
 - datasets tratados/AAK DO BRASIL CL 514.csv

Processando: 2RC IND EMBALAGENS.csv
Frequência detectada/definida: D
Início: 2024-03-01 | Fim: 2025-08-31 | n=549
ARIMA: Frequência Diária. Forçando sazonalidade semanal (m=7, D=1).

------------------------------
Treinando AUTO-ARIMA.


/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: F

ARIMA(p,d,q)x(P,D,Q)m selecionado: (1, 0, 2)x(0, 1, 1, 7)
RMSE ARIMA: 1.4531
MAE ARIMA: 1.0807
Accuracy ARIMA: -180.84%

------------------------------
Treinando LSTM...


/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Modelo LSTM salvo em: ./graficos_separados_90s/2RC IND EMBALAGENS_lstm.h5
RMSE LSTM: 1.2427
MAE LSTM: 0.9625
Accuracy LSTM: -705.29%

------------------------------
Treinando GRU...


...

/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: F

ARIMA(p,d,q)x(P,D,Q)m selecionado: (2, 0, 1)x(0, 1, 1, 7)
RMSE ARIMA: 5.0848
MAE ARIMA: 3.5905
Accuracy ARIMA: 84.48%

------------------------------
Treinando LSTM...


/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/Users/arielportela/Documents/auren/venv/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Modelo LSTM salvo em: ./graficos_separados_90s/AAK DO BRASIL CL 514_lstm.h5
RMSE LSTM: 4.7756
MAE LSTM: 3.0556
Accuracy LSTM: 86.25%

------------------------------
Treinando GRU...


Modelo GRU salvo em: ./graficos_separados_90s/AAK DO BRASIL CL 514_gru.h5
RMSE GRU: 5.6002
MAE GRU: 3.7773
Accuracy GRU: 84.95%

=== Tabela de Métricas ===
                     csv freq  RMSE_ARIMA  MAE_ARIMA MAPE_ARIMA ACCURACY_ARIMA (%)  RMSE_LSTM  MAE_LSTM MAPE_LSTM ACCURACY_LSTM (%)  RMSE_GRU  MAE_GRU MAPE_GRU ACCURACY_GRU (%)  test_samples_common
  2RC IND EMBALAGENS.csv    D    1.453092   1.080708     2.8084           -180.84%   1.242701  0.962512    8.0529          -705.29%  1.603526 1.211734   9.9097         -890.97%                   48
   AACD - IBIRAPUERA.csv    D    2.596874   2.442360     0.1813             81.87%   1.784704  1.285110    0.1038            89.62%  0.863095 0.669467   0.0502           94.98%                  103
AAK DO BRASIL CL 514.csv    D    5.084823   3.590490     0.1552             84.48%   4.775631  3.055583    0.1375            86.25%  5.600176 3.777345   0.1505           84.95%                  206

Arquivo de métricas completo salvo em: ./graficos_s